# Fase 0 — Arqueología comparativa

**Proyecto:** CRM-Granos-MX  
**Fecha:** 2026-04-30  
**Autor:** Claude Code (delegado por el operador del negocio)

## Propósito de este notebook

El prompt maestro v2 nos pide "hacer arqueología" del CRM existente antes de proponer nada. Como decidimos que **CRM-Granos-MX es proyecto nuevo independiente** (no refactor del legacy), aquí la arqueología es **comparativa**: catalogo qué cosas del proyecto vecino [`intergranel-iq`](../../intergranel-iq/) sirven como referencia para no repetir errores, y qué cosas se descartan explícitamente.

Este notebook es vivo — al cerrar Fase 1 lo extiendo con visualizaciones de cobertura una vez tengamos data real cargada en Postgres.

## 1. El proyecto referencia: `intergranel-iq`

**Lo que es:** sistema legacy de inteligencia de mercado (precios CBOT, episodios históricos, alertas, RAG sobre maíz) **+** un módulo de prospección comercial dentro (`scripts/prospeccion/`) que era el predecesor de CRM-Granos-MX.

**Estado:** vivo en producción. Auditoría exhaustiva del 2026-04-26 disponible en [`intergranel-iq/AUDITORIA_2026-04-26.md`](../../intergranel-iq/AUDITORIA_2026-04-26.md).

**Decisión clave del usuario (2026-04-29):** *"Es un nuevo proyecto totalmente. El anterior no cambia nada. Son proyectos independientes. Te puedes inspirar y tomar de ahí información, pero el otro proyecto se mantiene tal cual."*

Por eso este notebook NO toca `intergranel.db`, NO importa código del viejo, NO comparte stack con él. Solo es referencia.

## 2. Inventario comparativo — qué tomamos, qué descartamos, qué nuevo

| Elemento del legacy | Estado del legacy (auditoría 2026-04-26) | Decisión para CRM-Granos-MX | Por qué |
|---|---|---|---|
| **Stack: Flask + SQLite** | maduro pero no escala (28k filas, queries lentas) | **PostgreSQL + PostGIS + FastAPI** | spatial joins reales, async I/O, rendimiento bajo carga, prod-ready |
| **Schema 80 columnas en `prospectos`** | 40+ columnas al 100% NULL | **schema delgado, columnas se agregan cuando se llenen** | evitamos sobre-diseño |
| **2 SCIAN objetivo (311830, 311211)** | cobertura limitada al canal tortillerías | **8 SCIAN en 4 canales** | replica el modelo de negocio dual real |
| **2 estados cargados (CDMX, EdoMex) + 2 fuera de plan (Gto, Qro)** | foco accidental | **13 entidades centro-sureste, descarga limpia** | alineado con plan estratégico |
| **Detección de cadenas con fuzzy matching** | funciona, ~2,067 cadenas detectadas | **se conserva la lógica, se reescribe en `enrichment/matching.py`** | técnica probada |
| **Scoring 0-100, 6 componentes** | calibrado solo a tortillerías | **scoring por canal, pesos distintos por canal** | granjas y forrajeras tienen drivers diferentes a tortillerías |
| **Dashboard Leaflet self-contained HTML** | funciona, pero no compartible remotamente | **dashboard web FastAPI + frontend separado** | el equipo trabaja remoto |
| **Auth con 3 roles + login + recovery** | funciona bien | **se replica el modelo (3 roles, JWT, bcrypt)** | sirve, no hay que reinventar |
| **Migraciones ad-hoc + Alembic baseline tarde** | deuda técnica reconocida | **Alembic desde día 1**, raw SQL `op.execute()` | cualquier cambio rastreable |
| **Pipeline Kanban React local-only** | el Drawer no persistía al backend (bug crítico parchado tarde) | **persistencia siempre via API REST**, frontend nunca con estado autoritativo | regla de oro: server is source of truth |
| **Sin histórico de etapas pipeline** | no se sabe cuánto duró un prospecto en cada etapa | **tabla `pipeline_etapas_historial` desde día 1** | trazabilidad operacional |
| **1 contacto por prospecto (4 columnas planas)** | si hay 3 personas relevantes, sólo cabe 1 | **modelo extensible** (interacciones por contacto distinto en `interacciones.contacto_persona`) | comité de compras |
| **Sin compliance LFPDPPP estructurado** | columnas PII no marcadas | **catálogo `_columnas_pii` + bitácora `compliance_log`** | autoridad reformada en Mar-2025, multas 320,000 UMAs |
| **Sin cruce con SAT 69-B** | riesgo de prospectar contribuyentes en lista negra | **`sat_lista_69b` + cruce mensual obligatorio** | due diligence fiscal |
| **Sin AGEB / NSE** | scoring sin contexto socioeconómico | **`agebs` + `indicadores_geograficos`** | mejor scoring de capacidad de compra |
| **Handoff a `comercial-app`** | endpoint `api_prospecto_convertir` (5 prospectos convertidos) | **NO se integra al `comercial-app` v1** | proyecto nuevo paralelo ("comercial-app v2") es decisión futura |
| **Inteligencia de mercado (precios, RAG, episodios)** | corazón del legacy, funciona | **se queda en `intergranel-iq`**, CRM-Granos-MX no la replica | bounded contexts |

## 3. Dimensionamiento del universo (DENUE Cuantificar, 2026-04-30)

Cargamos los conteos generados durante la validación del token INEGI.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd().parent
DIM = ROOT / "data" / "raw" / "dimensionamiento_inicial.json"
if DIM.exists():
    dim = json.loads(DIM.read_text())
    print(f"Fuente: {dim['fuente']}")
    print(f"Fecha: {dim['fecha']}")
    print(f"\nTotal en MX (5 SCIAN primarios): {dim['total_mx_5_scian']:,}")
    print(f"Total en 13 priorizados:         {dim['total_priorizados_5_scian']:,}")
    print(f"\nPor canal:")
    for canal, valores in dim['por_canal'].items():
        print(f"  {canal:<25} MX: {valores['mx']:>10,}   priorizados: {valores['priorizados']:>10,}")
    print(f"\nDistribución por entidad priorizada:")
    for cve, total in dim['por_entidad_priorizada'].items():
        print(f"  {cve}: {total:>10,}")
else:
    print(f'data/raw/dimensionamiento_inicial.json no encontrado.\nGenéralo corriendo el snippet de validación INEGI documentado en CLAUDE.md §9.')

## 4. Hallazgos clave durante la validación

Estas correcciones al prompt maestro v2 ya están aplicadas en el código y documentadas en `CLAUDE.md` §9.

### 4.1 URL del DENUE estaba obsoleta
El prompt cita `https://www.inegi.org.mx/app/api/denue/v1/Cuantificar/...`. Ese endpoint devuelve HTTP 200 pero el body es página de error "Esta liga ya no existe". La URL correcta es `https://www.inegi.org.mx/app/api/denue/v1/consulta/Cuantificar/...` (con `/consulta/` adicional).

### 4.2 SCIAN del prompt para alimento balanceado son inválidos
El prompt lista `311111` y `311119`. Ninguno existe en SCIAN 2018 (que es el que INEGI usa). El correcto es `311110` — "Elaboración de alimentos para animales". Total nacional: 865 establecimientos.

### 4.3 SCIAN 461110 (abarrotes menudeo) es ruido
387,149 establecimientos en las 13 entidades priorizadas. Son las tienditas de la esquina. Excluido de descarga primaria; queda como cruce on-demand en Fase 5.

### 4.4 Granjas pecuarias NO están en DENUE
SCIAN 1121-1129 (avicultura, porcicultura, bovinos, ovinos, otros) devuelven 0 en DENUE. Son sector primario, viven en SENASICA / SIAP / Censo Agropecuario INEGI. Se cubren en Fase 5 con fuentes alternativas.

### 4.5 Forrajeras de pueblo se registran bajo SCIAN 434112
No es uno de los SCIAN obvios — es "Comercio al por mayor de medicamentos veterinarios y alimentos para animales (excepto mascotas)". Encontrado descodificando el CLEE de búsquedas DENUE por nombre "forrajera". 7,655 establecimientos en las 13 entidades, concentrados en EdoMex, Puebla, Veracruz.

### 4.6 Asociaciones ganaderas locales sí están en DENUE bajo 813110
2,043 establecimientos en las 13 entidades. Son canal B2B2C valioso: vendes a la asociación, ellos venden/distribuyen a sus 50-300 socios ganaderos.

## 5. Riesgos identificados (mantenidos también en `CLAUDE.md` §10)

### Técnicos
- **Google Drive sync vs git.** Operaciones git largas pueden colisionar con Drive y corromper. Mitigación: si vemos errores raros en git, sospecharlo primero.
- **Postgres.app vs Render version drift.** Postgres.app trae 17 latest, Render usa 16. Funciones marginales pueden diferir. Mitigación: tests CI contra Postgres 16.
- **Token INEGI rate limit.** 60 req/min recomendado. Descarga nacional con paginación tomará 4-6 horas. Mitigación: throttling con tenacity, persistencia incremental.

### Compliance
- **Aviso de privacidad no existe todavía.** Crítico antes del primer contacto comercial real (Fase 8). Genera en Fase 7.
- **`razon_social` y `nombre` pueden ser PF.** Marcados en `_columnas_pii` desde la migración baseline. Cualquier export sin filtrar PII rompe LFPDPPP.
- **Token INEGI en chat de Claude.** El usuario lo pasó plano. Sesión privada pero rotarlo ante duda.

### Proyecto
- **Cambio de SCIAN del prompt.** Cualquier doc externo (slide, propuesta) que cite los 8 SCIAN del prompt requiere actualizar.
- **Granjas integradas no se cubren en Fases 3-4.** Si alguien pregunta "¿y Bachoco granjas?", la respuesta hoy es "Fase 5".
- **CANAMI sitio no responde.** Investigar fuentes alternativas en Fase 5.

## 6. Próximos pasos

Al cerrar Fase 0:
1. Instalar Postgres.app (Fase 0.E — interactivo).
2. Crear DB `crm_granos_mx`, habilitar PostGIS + pg_trgm.
3. `alembic upgrade head` aplica la migración baseline.
4. Verificar que `_columnas_pii` quedó poblado con las 21 columnas conocidas.
5. Commit final de Fase 0 y reporte ejecutivo en chat.

Fase 1 — Reporte de reconciliación: comparar lo que tenemos contra el plan de las 8 fases del prompt y proponer ajustes accionables.

Fase 2 — Auditoría de la data DENUE existente: como aquí no heredamos data del legacy, esta fase se reduce a definir los chequeos de calidad para aplicar a la primera descarga real (Fase 3).